# Challenge 1 - T-test

In statistics, t-test is used to test if two data samples have a significant difference between their means. There are two types of t-test:

* **Student's t-test** (a.k.a. independent or uncorrelated t-test). This type of t-test is to compare the samples of **two independent populations** (e.g. test scores of students in two different classes). `scipy` provides the [`ttest_ind`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_ind.html) method to conduct student's t-test.

* **Paired t-test** (a.k.a. dependent or correlated t-test). This type of t-test is to compare the samples of **the same population** (e.g. scores of different tests of students in the same class). `scipy` provides the [`ttest_re`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_rel.html) method to conduct paired t-test.

Both types of t-tests return a number which is called the **p-value**. If p-value is below 0.05, we can confidently declare the null-hypothesis is rejected and the difference is significant. If p-value is between 0.05 and 0.1, we may also declare the null-hypothesis is rejected but we are not highly confident. If p-value is above 0.1 we do not reject the null-hypothesis.

Read more about the t-test in [this article](https://researchbasics.education.uconn.edu/t-test/) and [this Quora](https://www.quora.com/What-is-the-difference-between-a-paired-and-unpaired-t-test). Make sure you understand when to use which type of t-test. 

## Example: Testing Coffee's Effect on Heart Rate

Here's an example. You want to see if coffee changes heart rate. There are two ways to perform an experiment:

1. **Paired**: You take 10 people and record their heart rate before drinking a cup of coffee. Let's say \( $y_{i1}$ \) is the pre-coffee heart rate of the ith person. They drink the coffee, and then you record the same person's heart rate again, \( $y_{i2}$ \). Each person has two measurements recorded, and the difference in heart rate for each person is \( $d_{i}  = y_{i2} - y_{i1}$ \). You can then perform a one-sample t-test on the differences (\( $d_{1}, ... , d_{10} $\)) to test the null hypothesis \( $H_{0}: delta = 0$ \), where \( delta \) represents the population change in heart rate after drinking coffee. The estimated difference is \( $\bar{x}$ - $\bar{y}$ \), the sample mean of the differences.

2. **Unpaired**: You take 10 subjects and record their heart rate before drinking coffee, yielding measurements \( $ y_{1}, \ldots, y_{10} $ \). You take 15 *different* people and record their heart rate after they drink a cup of coffee, obtaining measurements \( $ x_{1}, \ldots, x_{15} $ \). The estimated difference in means is \( $ \bar{x} - \bar{y} $ \), comparing the two sample means. To test the null hypothesis \( $ H_{0}: \delta = 0 $\), you perform a two-sample t-test.

Paired analyses can be more efficient than unpaired analyses because each person serves as their own control. People's heart rates vary naturally regardless of coffee consumption, meaning the unpaired t-test can lose efficiency by ignoring this person-to-person variability.

In [56]:
# Import libraries

import pandas as pd
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel

#### Import dataset

In this challenge we will work on the Pokemon dataset you have used last week. The goal is to test whether different groups of pokemon (e.g. Legendary vs Normal, Generation 1 vs 2, single-type vs dual-type) have different stats (e.g. HP, Attack, Defense, etc.).

In [8]:
# Import dataset

pokemon = pd.read_csv('../data/Pokemon.csv')

pokemon.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


#### First we want to define a function with which we can test the means of a feature set of two samples. 

In the next cell you'll see the annotations of the Python function that explains what this function does and its arguments and returned value. This type of annotation is called **docstring** which is a convention used among Python developers. The docstring convention allows developers to write consistent tech documentations for their codes so that others can read. It also allows some websites to automatically parse the docstrings and display user-friendly documentations.

Follow the specifications of the docstring and complete the function.

In [ ]:
def t_test_features(s1, s2, features=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Total']):
    """Test means of a feature set of two samples
    
    Args:
        s1 (dataframe): sample 1
        s2 (dataframe): sample 2
        features (list): an array of features to test
    
    Returns:
        dict: a dictionary of t-test scores for each feature where the feature name is the key and the p-value is the value
    """
    def perform_ttest(feature):    
        p_value = float(ttest_ind(s1[feature], s2[feature], equal_var=False).pvalue)
        return feature, p_value
    results = dict(map(perform_ttest, features))
    return results


#### Using the `t_test_features` function, conduct t-test for Lengendary vs non-Legendary pokemons.

*Hint: your output should look like below:*

```
{'HP': 1.0026911708035284e-13,
 'Attack': 2.520372449236646e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239812e-21,
 'Sp. Def': 2.2949327864052826e-15,
 'Speed': 1.049016311882451e-18,
 'Total': 9.357954335957446e-47}
 ```

In [26]:
# Your code here
legendary_stats = pokemon[pokemon["Legendary"] == True]
normal_stats = pokemon[pokemon["Legendary"] == False]
p_features = t_test_features(legendary_stats,normal_stats)
p_features

{'HP': 1.002691170803531e-13,
 'Attack': 2.52037244923666e-16,
 'Defense': 4.82699849491934e-11,
 'Sp. Atk': 1.551461411223974e-21,
 'Sp. Def': 2.2949327864052925e-15,
 'Speed': 1.0490163118824546e-18,
 'Total': 9.357954335957505e-47}

#### From the test results above, what conclusion can you make? Do Legendary and non-Legendary pokemons have significantly different stats on each feature?

In [37]:
# Your comment here
alpha = 0.05
if all(val < alpha for val in p_features.values()):
    print('null hypothesis has been rejected for all the feautures:\nlegendary and non legendary pokemon differ in all features')
else:
    print('for at least one feauter the test fails to reject the null hypothesis')

null hypothesis has been rejected for all the feautures:
legendary and non legendary pokemon differ in all features


#### Next, conduct t-test for Generation 1 and Generation 2 pokemons.

In [ ]:
# Your code here
first_stats = pokemon[pokemon['Generation'] == 1]
second_stats = pokemon[pokemon['Generation'] == 2]
p_features = t_test_features(first_stats,second_stats)
print(p_features)


{'HP': 0.14551697834219646,
 'Attack': 0.2472195896721772,
 'Defense': 0.5677711011725424,
 'Sp. Atk': 0.12332165977104394,
 'Sp. Def': 0.18829872292645747,
 'Speed': 0.0023926593731213447,
 'Total': 0.5631377907941677}

#### What conclusions can you make?

In [43]:
# Your comment here
alpha = 0.05

if all(val < alpha for val in p_features.values()):
    print('null hypothesis has been rejected for all the feautures:\n1st and 2nd generation pokemon differ in all features')
if all(val > alpha for val in p_features.values()):
    print('null hypothesis cannot been rejected for all the feautures:\n1st and 2nd generation pokemon do not differ in any of the features')
else:
    for feature, p_value in p_features.items():
        if p_value < alpha:
            print(f"for {feature} null hypothesis has been rejected: 1st and 2nd generation pokemon have different {feature} stats")
        else:
            print(f"for {feature} null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal {feature} stats")

for HP null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal HP stats
for Attack null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal Attack stats
for Defense null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal Defense stats
for Sp. Atk null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal Sp. Atk stats
for Sp. Def null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal Sp. Def stats
for Speed null hypothesis has been rejected: 1st and 2nd generation pokemon have different Speed stats
for Total null hypothesis cannot be rejected: 1st and 2nd generation pokemon have equal Total stats


#### Compare pokemons who have single type vs those having two types.

In [52]:
# Your code here
one_stats = pokemon[pokemon['Type 2'].isna()]
two_stats = pokemon[~pokemon['Type 2'].isna()]
p_features = t_test_features(first_stats,second_stats)
print(p_features)

{'HP': 0.14551697834219646, 'Attack': 0.2472195896721772, 'Defense': 0.5677711011725424, 'Sp. Atk': 0.12332165977104394, 'Sp. Def': 0.18829872292645747, 'Speed': 0.0023926593731213447, 'Total': 0.5631377907941677}


#### What conclusions can you make?

In [55]:
# Your comment here

alpha = 0.05

if all(val < alpha for val in p_features.values()):
    print('null hypothesis has been rejected for all the feautures:\nOne-type and Two-type pokemon differ in all features')
if all(val > alpha for val in p_features.values()):
    print('null hypothesis cannot been rejected for all the feautures:\nOne-type and Two-type pokemon do not differ in any of the features')
else:
    for feature, p_value in p_features.items():
        if p_value < alpha:
            print(f"for {feature} null hypothesis has been rejected: One-type and Two-type pokemon have different {feature} stats")
        else:
            print(
                f"for {feature} null hypothesis cannot be rejected: One-type and Two-type pokemon have equal {feature} stats"
                )

for HP null hypothesis cannot be rejected: One-type and Two-type pokemon have equal HP stats
for Attack null hypothesis cannot be rejected: One-type and Two-type pokemon have equal Attack stats
for Defense null hypothesis cannot be rejected: One-type and Two-type pokemon have equal Defense stats
for Sp. Atk null hypothesis cannot be rejected: One-type and Two-type pokemon have equal Sp. Atk stats
for Sp. Def null hypothesis cannot be rejected: One-type and Two-type pokemon have equal Sp. Def stats
for Speed null hypothesis has been rejected: One-type and Two-type pokemon have different Speed stats
for Total null hypothesis cannot be rejected: One-type and Two-type pokemon have equal Total stats


#### Now, we want to compare whether there are significant differences of `Attack` vs `Defense`  and  `Sp. Atk` vs `Sp. Def` of all pokemons. Please write your code below.

*Hint: are you comparing different populations or the same population?*

In [57]:
# Your code here
def t_test_features(sample):
    """Test means of pair of features 
    
    Args:
        sample data: a dataframe
    
    Returns:
        a tuple of t-test scores for attack vs defense and Sp. Atk vs Sp. Def
    """
    p_atk_def = float(ttest_rel(sample['Attack'], sample['Defense']).pvalue)
    p_sptk_spdf = float(ttest_rel(sample['Sp. Atk'], sample['Sp. Def']).pvalue)
    return p_atk_def,p_sptk_spdf



In [58]:
t_test_features(pokemon)

(1.7140303479358582e-05, 0.3933685997548118)

#### What conclusions can you make?

In [61]:
# Your comment here
p_atk_def, p_sptk_spdf = t_test_features(pokemon)

if p_atk_def < alpha:
    print(f"null hypothesis has been rejected: attack and defense have different means")
else:
    print(f"null hypothesis cannot be rejected: attack and defense have equal means")

if p_sptk_spdf < alpha:
    print(f"null hypothesis has been rejected: sp. attack and sp. defense have different means")
else:
    print(f"null hypothesis cannot be rejected: sp. attack and sp. defense have equal means")


null hypothesis has been rejected: attack and defense have different means
null hypothesis cannot be rejected: sp. attack and sp. defense have equal means
